In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
from torchvision.utils import save_image

In [ ]:
model = models.vgg19(pretrained=True).features

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 187MB/s]


In [ ]:
print(model
)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (17): ReLU(inplace=True)
  (18): MaxPoo

In [ ]:
class VGG(nn.Module):
    def __init__(self):
        super(VGG, self).__init__()
        self.choosen_features = ['0', '5', '10', '19', '28']
        self.model = models.vgg19(pretrained=True).features[:29]

    def forward(self, x):
        features = []
        for layer_num, layer in enumerate(self.model):
            x = layer(x)
            if str(layer_num) in self.choosen_features:
                features.append(x)
        return features

In [ ]:
def load_image(image_name):
   image = Image.open(image_name)
   image = loader(image).unsqueeze(0)
   return image.to(device)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = 356

In [ ]:
loader = transforms.Compose(
    [
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ]
)

In [ ]:
model = VGG().to(device).eval()

In [ ]:
orignal_img = load_image('car.png')
style_img = load_image('style.png')

In [ ]:
genrated = orignal_img.clone().requires_grad_(True)

In [ ]:
total_steps = 6000
learning_rate = 0.001
alpha =1
beta = 0.01
optimizer = optim.Adam([genrated], lr=learning_rate)

In [ ]:
for step in range(total_steps):
    generated_features = model(genrated)
    original_img_features = model(orignal_img)
    style_features = model(style_img)

    style_loss = 0
    original_loss = 0

    for gen_feature, orig_feature, style_feature in zip(
        generated_features, original_img_features, style_features):

        batch_size, channel, height, width = gen_feature.shape


        original_loss += torch.mean((gen_feature - orig_feature) ** 2)

        G = gen_feature.view(channel, height * width).mm(
            gen_feature.view(channel, height * width).t()
        )

        A = style_feature.view(channel, height * width).mm(
            style_feature.view(channel, height * width).t()
        )

        style_loss += torch.mean((G - A) ** 2)

    total_loss = alpha * original_loss + beta * style_loss

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()


    if step % 200 == 0:
        print(total_loss)
        save_image(genrated,"genrated.png")


tensor(7270883.5000, device='cuda:0', grad_fn=<AddBackward0>)
tensor(857586., device='cuda:0', grad_fn=<AddBackward0>)
tensor(396026.2188, device='cuda:0', grad_fn=<AddBackward0>)
tensor(242863.5312, device='cuda:0', grad_fn=<AddBackward0>)
tensor(166425.0469, device='cuda:0', grad_fn=<AddBackward0>)
tensor(121900.1172, device='cuda:0', grad_fn=<AddBackward0>)
tensor(93937.8281, device='cuda:0', grad_fn=<AddBackward0>)
tensor(75024.3438, device='cuda:0', grad_fn=<AddBackward0>)
tensor(61546.0117, device='cuda:0', grad_fn=<AddBackward0>)
tensor(51660.0117, device='cuda:0', grad_fn=<AddBackward0>)
tensor(44186.0234, device='cuda:0', grad_fn=<AddBackward0>)
tensor(38425.4102, device='cuda:0', grad_fn=<AddBackward0>)
tensor(33854.5859, device='cuda:0', grad_fn=<AddBackward0>)
tensor(30143.7227, device='cuda:0', grad_fn=<AddBackward0>)
tensor(27080.9453, device='cuda:0', grad_fn=<AddBackward0>)
tensor(24548.1914, device='cuda:0', grad_fn=<AddBackward0>)
tensor(22423.0469, device='cuda:0', g